<a href="https://colab.research.google.com/github/CodingKoopa/Robot2020/blob/master/assign1_loerker_B00908992.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
from collections.abc import Callable
from typing import Optional

import logging
import math
import random
import sys
import unittest

import scipy.special

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

linear_function = lambda x: x
transcendental_function = lambda x: (x * math.exp(x) + x)
gamma_function = lambda x: scipy.special.gamma(x)

RUN_TESTS = True

In [12]:
def find_bounds(f: Callable[[float], float], y: float) -> tuple[float, float]:
  # Whether to search positive or negative x-values.
  positive_x = f(0) < y
  logger.debug(f"{f(0)=} {y=} {positive_x=}")

  smaller_bound_candidate = 0
  larger_bound_candidate = float(2 if positive_x else -2)
  while True:
    logger.debug(f"Upper/lower bound candidate: {larger_bound_candidate}")
    # If we are searching for an upper bound.
    if positive_x:
      if f(larger_bound_candidate) >= y:
        break
    # If we are searching for a lower bound.
    else:
      if f(larger_bound_candidate) <= y:
        break

    # Move up the smaller bound.
    smaller_bound_candidate = larger_bound_candidate
    # Move up the larger bound.
    larger_bound_candidate *= 2
    if math.isinf(larger_bound_candidate):
      if positive_x:
        larger_bound_candidate = sys.float_info.max
        assert f(larger_bound_candidate) >= y
      else:
        larger_bound_candidate = sys.float_info.min
        assert f(larger_bound_candidate) <= y
      break
  logger.debug(f"Inclusive bounds to search: ({smaller_bound_candidate}, {larger_bound_candidate})")
  assert abs(smaller_bound_candidate) < abs(larger_bound_candidate)
  return ((smaller_bound_candidate, larger_bound_candidate) if positive_x
    else (larger_bound_candidate, smaller_bound_candidate))

class FindBoundTest(unittest.TestCase):
  def test_inf(self):
    with self.assertRaises(AssertionError):
      find_bounds(linear_function, math.inf)

  def test_bigposnum(self):
    self.assertEqual(find_bounds(linear_function, 99999999999), (68719476736.0, 137438953472.0))

  def test_bignegnum(self):
    self.assertEqual(find_bounds(linear_function, -99999999999), (-137438953472.0, -68719476736.0))

  def test_trans(self):
    self.assertEqual(find_bounds(transcendental_function, 5000), (4.0, 8.0))

if __name__ == "__main__" and RUN_TESTS:
  unittest.main(argv=['', 'FindBoundTest'], exit=False)
  pass

....
----------------------------------------------------------------------
Ran 4 tests in 0.012s

OK


In [31]:
# Referenced the discrete version: https://en.wikipedia.org/wiki/Binary_search#Procedure
def binary_search(f: Callable[[float], float], target_y: float, search_interval: tuple[float, float]) -> float:
  lower = search_interval[0]
  upper = search_interval[1]
  logger.debug(f"Binary searching for {target_y} in ({lower},{upper}).")
  while lower <= upper:
    center_x = (lower + upper) / 2
    center_y = f(center_x)
    logger.debug(f"{lower=} {center_x=} {upper=} ({center_y=})")
    if center_y < target_y:
      lower = math.nextafter(center_x, math.inf)
      logger.debug(f"{lower=}")
    elif center_y > target_y:
      upper = math.nextafter(center_x, -math.inf)
      logger.debug(f"{upper=}")
    else:
      # We're spot on!
      return center_x
  # This is close enough.
  return center_x

def invert(f: Callable[[float], float], desired_y: float, increasing_interval: Optional[tuple[float, float]] = None):
  search_interval = increasing_interval
  if search_interval is None:
    # Pick a side of the y-axis to search on.
    search_interval = find_bounds(f, desired_y)
  x_value = binary_search(f, desired_y, search_interval)
  actual_y = f(x_value)
  diff = abs(actual_y - desired_y)
  if diff != 0:
    logger.warning(f"Inversion is off by {diff} ({desired_y=}, {actual_y=})")
  return x_value

In [33]:
class InvertTest(unittest.TestCase):
  def invert(self, f, increasing_interval = None):
    minimum_y = f(increasing_interval[0]) if increasing_interval is not None else -math.inf
    maximum_y = f(increasing_interval[1]) if increasing_interval is not None else math.inf
    logger.info(f"Testing zero...")
    if (minimum_y < 0 and maximum_y > 0):
      invert(f, 0, increasing_interval)
    logger.info(f"Testing small positive values...")
    i = 1
    while True:
      y = float(2) ** -i
      if y == 0 or not (minimum_y <= y <= maximum_y):
        break;
      invert(f, y, increasing_interval)
      i += 1
    logger.info(f"Testing small negative values...")
    i = 1
    while True:
      y = float(-2) ** -i
      if y == 0 or not (minimum_y <= y <= maximum_y):
        break;
      invert(f, y, increasing_interval)
      i += 1
    logger.info(f"Testing large positive values...")
    i = 1
    while True:
      y = float(2) ** i
      if y == math.inf or not (minimum_y <= y <= maximum_y):
        break;
      try:
        invert(f, y, increasing_interval)
      except OverflowError as e:
        logger.warning(f"Overflow while computing f^-1({y=}) (this is probably fine)")
        break
      i += 1
    logger.info(f"Testing large negative values...")
    i = 1
    while True:
      y = float(-2) ** i
      if y == -math.inf or not (minimum_y <= y <= maximum_y):
        break;
      try:
        invert(f, y, increasing_interval)
      except OverflowError as e:
        logger.warning(f"Overflow while computing f^-1({y=}) (this is probably fine)")
        break
      i += 1

  def test_trans(self):
    logger.info("TRANSCENDENTAL FUNCTION TEST")
    self.invert(transcendental_function)

  def test_gamma(self):
    logger.info("GAMMA FUNCTION TEST")
    self.invert(gamma_function, (1.5, 16.0))

if __name__ == "__main__":
  unittest.main(argv=['', 'InvertTest'], exit=False)

INFO:__main__:GAMMA FUNCTION TEST
INFO:__main__:Testing zero...
INFO:__main__:Testing small positive values...
INFO:__main__:Testing small negative values...
INFO:__main__:Testing large positive values...
INFO:__main__:Testing large negative values...
.INFO:__main__:TRANSCENDENTAL FUNCTION TEST
INFO:__main__:Testing zero...
INFO:__main__:Testing small positive values...
INFO:__main__:Testing small negative values...
INFO:__main__:Testing large positive values...
INFO:__main__:Testing large negative values...
.
----------------------------------------------------------------------
Ran 2 tests in 19.507s

OK


In [56]:
question2_function = lambda x: math.sin(x) / x

def maxmin_value(f: Callable[[float], float], range: tuple[float, float],
                 cmp: Callable[[float, float], bool]):
  STEP_SIZE = 0.01
  assert range[0] <= range[1]
  cursor = range[0]
  ret = f(cursor)
  while cursor <= range[1]:
    cursor += STEP_SIZE
    ret = cmp(ret, f(cursor))
  return ret

def min_value(f: Callable[[float], float], range: tuple[float, float]):
  return maxmin_value(f, range, min)

def max_value(f: Callable[[float], float], range: tuple[float, float]):
  return maxmin_value(f, range, max)

class MaxMinValueTest(unittest.TestCase):
  def test_q2max(self):
    self.assertEqual(max_value(question2_function, (-1, 1)), 1.0)

  def test_q2min(self):
    self.assertAlmostEqual(min_value(question2_function, (-1, 1)), 0.84147, places=5)

# Referenced https://en.wikipedia.org/wiki/Monte_Carlo_integration (but avoided looking at pseudocode).
def integrate(f: Callable[[float], float], x_range: tuple[float, float]):
  NUM_ATTEMPTS = 9999999

  f_max = max(0.0, max_value(f, x_range))
  f_min = min(0.0, max_value(f, x_range))
  area = (x_range[1] - x_range[0]) * (f_max - f_min)
  logger.info(f"Approximate bounding box: [{x_range[0]}, {f_min}] to [{x_range[1]}, {f_max}] (area {area})")

  num_hits = 0
  for _ in range(NUM_ATTEMPTS):
    point_x = random.uniform(x_range[0], x_range[1])
    point_y = random.uniform(f_min, f_max)
    f_y = f(point_x)
    if point_y <= f_y:
      num_hits += 1

  return (num_hits / NUM_ATTEMPTS) * area

if __name__ == "__main__"  and RUN_TESTS:
  unittest.main(argv=['', 'MaxMinValueTest'], exit=False)
  pass

..
----------------------------------------------------------------------
Ran 2 tests in 0.004s

OK


In [58]:
class IntegrateTest(unittest.TestCase):
  def test_integrate(self):
    self.assertAlmostEqual(integrate(question2_function, (-1, 1)), 1.8921661407, places=4)

if __name__ == "__main__":
  unittest.main(argv=['', 'IntegrateTest'], exit=False)
  pass

INFO:__main__:Approximate bounding box: [-1, 0.0] to [1, 1.0] (area 2.0)
.
----------------------------------------------------------------------
Ran 1 test in 10.551s

OK
